# Domain 4: Prompt Engineering & Structured Output
**Exam**: Claude Certified Architect – Foundations (CCAR-F)
**Weighting**: 20% of exam
**Depth**: deep

## Domain Overview

This domain covers designing prompts with explicit criteria to reduce false positives, applying
few-shot prompting, enforcing structured output via `tool_use` and JSON schemas, implementing
validation-retry loops, designing batch processing strategies, and multi-instance/multi-pass review
architectures.

**Why it matters**: reliable structured extraction and low-false-positive review systems are core
production patterns tested at 20% weight — architects must know both the prompting techniques and
their limits.

**Key concepts to know**: explicit vs. vague criteria, few-shot example design, `tool_use` +
`tool_choice` for guaranteed schema compliance, retry-with-feedback loops, the Message Batches API,
and independent-review patterns.

In [ ]:
# Setup: these notebooks demonstrate Claude Agent SDK / Claude API / MCP concepts using
# plain Python simulations of the relevant objects and control flow, so every cell is runnable
# without needing a live Claude Agent SDK installation or API key. Where call_claude() is used,
# it falls back to a simulated response if ANTHROPIC_API_KEY is not set.
import os

def call_claude(prompt, system=None, model="claude-sonnet-4-5", max_tokens=600):
    try:
        import anthropic
        api_key = os.environ.get("ANTHROPIC_API_KEY")
        if not api_key:
            raise RuntimeError("no ANTHROPIC_API_KEY set")
        client = anthropic.Anthropic(api_key=api_key)
        kwargs = {"model": model, "max_tokens": max_tokens, "messages": [{"role": "user", "content": prompt}]}
        if system:
            kwargs["system"] = system
        resp = client.messages.create(**kwargs)
        return resp.content[0].text
    except Exception:
        return f"[SIMULATED RESPONSE - set ANTHROPIC_API_KEY to call the real API] Prompt: {prompt[:120]}..."

print("call_claude() ready. Concepts below use plain Python to simulate Agent SDK / MCP / Claude Code behavior.")

## Key Concepts

### Design prompts with explicit criteria to improve precision and reduce false positives

Explicit, categorical criteria ("flag comments only when claimed behavior contradicts actual
code behavior") outperform vague instructions ("be conservative", "only report high-confidence
findings") at reducing false positives. High false-positive categories should be temporarily disabled
while their prompts are improved, to protect trust in the accurate categories.

In [ ]:
vague_instruction = "Only flag issues you're confident about."
explicit_instruction = (
    "Flag a comment as INCORRECT only if it makes a factual claim about behavior that "
    "directly contradicts the code beneath it (e.g., comment says 'always returns None' "
    "but the code has a return statement with a value). Do NOT flag style, phrasing, or "
    "comments that are merely incomplete."
)
print(len(vague_instruction), "chars (vague) vs", len(explicit_instruction), "chars (explicit, testable)")

### Apply few-shot prompting to improve output consistency and quality

Few-shot examples are the most effective technique for consistent formatting and for
demonstrating ambiguous-case handling. 2-4 targeted examples that show reasoning for a choice
(not just the answer) help the model generalize judgment to novel patterns, and reduce
hallucination in extraction tasks with varied document structures.

In [ ]:
few_shot_examples = """
Example 1:
Input: "Refund request, order #55, customer says item never arrived."
Output: {"category": "shipping_issue", "priority": "high", "reasoning": "Non-delivery claims risk churn."}

Example 2:
Input: "Question about how to change my email address."
Output: {"category": "account_settings", "priority": "low", "reasoning": "Self-service, no urgency."}
"""
new_input = "Charged twice for the same order."
print(call_claude(f"{few_shot_examples}\nNow classify:\nInput: \"{new_input}\"\nOutput:"))

### Enforce structured output using tool use and JSON schemas

`tool_use` with a JSON schema is the most reliable way to guarantee schema-compliant
structured output, eliminating syntax errors (though not semantic errors). `tool_choice: "any"`
guarantees a tool call when the document type is unknown; a forced named tool ensures a specific
extraction runs first. Nullable fields prevent the model from fabricating values for information
that may not exist in the source.

In [ ]:
extraction_schema = {
    "name": "extract_invoice",
    "input_schema": {
        "type": "object",
        "properties": {
            "invoice_number": {"type": "string"},
            "total": {"type": "number"},
            "due_date": {"type": ["string", "null"]},  # nullable: don't fabricate if absent
            "category": {"type": "string", "enum": ["utilities", "software", "other"]},
        },
        "required": ["invoice_number", "total"],  # due_date/category can be optional depending on design
    },
}
import json
print(json.dumps(extraction_schema, indent=2))

### Implement validation, retry, and feedback loops for extraction quality

On validation failure, send a follow-up request including the original document, the failed
extraction, and the specific validation error, so the model can self-correct. Retries succeed for
format/structural errors but are ineffective when the needed information is simply absent from the
source document.

In [ ]:
def retry_with_feedback(document: str, failed_extraction: dict, validation_error: str) -> str:
    return call_claude(f"""Document: {document}
Failed extraction: {failed_extraction}
Validation error: {validation_error}
Correct the extraction to fix this specific error.""")

print(retry_with_feedback(
    "Invoice #INV-99, Total: $450.00",
    {"invoice_number": "INV-99", "total": "450.00"},  # total should be a number, not a string
    "total must be a JSON number, not a string",
))

### Design efficient batch processing strategies

The Message Batches API offers 50% cost savings with up to a 24-hour processing window and
no guaranteed latency SLA — suited to non-blocking, latency-tolerant workloads (overnight reports),
not blocking pre-merge checks. It does not support multi-turn tool calling within a single request.
`custom_id` correlates request/response pairs for resubmitting only failed documents.

In [ ]:
def fits_batch_api(is_blocking: bool, latency_tolerant: bool) -> bool:
    return (not is_blocking) and latency_tolerant

print(fits_batch_api(is_blocking=True, latency_tolerant=False))   # pre-merge check: no
print(fits_batch_api(is_blocking=False, latency_tolerant=True))   # overnight report: yes

batch_requests = [{"custom_id": f"doc-{i}", "params": {"document": f"content {i}"}} for i in range(3)]
print(batch_requests)

### Design multi-instance and multi-pass review architectures

A model retains reasoning context from generation, making self-review less effective at
catching its own issues than an independent instance without that context. Large multi-file reviews
benefit from splitting into per-file local-issue passes plus a separate cross-file integration pass to
avoid attention dilution and contradictory findings.

In [ ]:
def review_architecture(files_count: int, self_review: bool) -> str:
    if self_review:
        return "weak: same session retains reasoning context, less likely to question its own decisions"
    if files_count > 5:
        return "split into per-file passes + one cross-file integration pass"
    return "single independent review pass is sufficient"

print(review_architecture(files_count=14, self_review=False))
print(review_architecture(files_count=1, self_review=True))

## Mini Hands-On Project

**Covers**: explicit criteria, few-shot examples, structured output, validation-retry, batching.

**Scenario**: Build a structured data extraction pipeline for invoices with validation and retry.

### Step 1: Define the schema and few-shot examples

In [ ]:
invoice_schema = extraction_schema  # reuse from concept cell
few_shot = """
Example: Input: "Invoice #A-1, Total $99.00, due 2025-10-01" ->
Output: {"invoice_number": "A-1", "total": 99.00, "due_date": "2025-10-01", "category": "other"}
"""
print(few_shot)

### Step 2: Simulate an extraction and validate it

In [ ]:
extracted = {"invoice_number": "INV-77", "total": "not-a-number", "due_date": None, "category": "software"}

def validate_extraction(extraction: dict) -> str | None:
    if not isinstance(extraction.get("total"), (int, float)):
        return "total must be a JSON number, not a string"
    return None

error = validate_extraction(extracted)
print(error)

### Step 3: Retry with feedback if invalid

In [ ]:
if error:
    corrected = retry_with_feedback("Invoice #INV-77, Total: not-a-number, due: n/a", extracted, error)
    print(corrected)

## Lab Exercises

### Exercise 1: Vague-to-explicit criteria rewrite

**Objective**: Rewrite 2 vague review criteria into explicit, testable ones.

**Produce**: 2 (vague, explicit) pairs.

In [ ]:
vague_criteria = [
    "Flag anything that looks off.",
    "Only report serious problems.",
]
# TODO: rewrite each as an explicit, categorical, testable criterion
explicit_criteria = []
for v in vague_criteria:
    explicit_criteria.append("...")  # TODO
for v, e in zip(vague_criteria, explicit_criteria):
    print(v, "->", e)

# Expected output: 2 specific, testable criteria with concrete examples of what qualifies.

In [ ]:
# STRETCH GOAL: test both the vague and explicit versions against the same ambiguous input
# and compare which produces a more consistent/justifiable decision.
ambiguous_input = "This function has no error handling but has worked fine in production for 2 years."
print(call_claude(f"{explicit_criteria[0]}\nEvaluate: {ambiguous_input}"))

# Expected output: A more consistent, well-justified decision from the explicit-criteria version.

### Exercise 2: Few-shot example design

**Objective**: Write 3 few-shot examples for classifying support tickets by urgency, including reasoning.

**Produce**: 3 examples plus a test on a new unseen ticket.

In [ ]:
# TODO: write 3 few-shot examples (input -> {urgency, reasoning}) covering distinct urgency levels
few_shot_urgency = """
Example 1: ...
Example 2: ...
Example 3: ...
"""
new_ticket = "My account was charged $0 but I never placed an order."
print(call_claude(f"{few_shot_urgency}\nClassify: {new_ticket}"))

# Expected output: 3 well-differentiated examples with reasoning, and a sensible classification of the new ticket.

In [ ]:
# STRETCH GOAL: add a 4th example specifically demonstrating an ambiguous edge case and its resolution.
edge_case_example = "..."  # TODO
print(edge_case_example)

# Expected output: An edge-case example that clarifies handling of a genuinely ambiguous ticket.

### Exercise 3: Schema design with nullable/enum fields

**Objective**: Design a JSON schema for extracting event details with proper nullable and enum fields.

**Produce**: A schema dict and a test showing a missing field returns null, not a fabricated value.

In [ ]:
# TODO: design a schema for {event_name, date, location, category} where location may be absent
# (nullable) and category is an enum of ["conference","webinar","meetup","other"]
event_schema = {}  # TODO

print(event_schema)

# Expected output: A schema with location as nullable and category as a constrained enum with an 'other' fallback.

In [ ]:
# STRETCH GOAL: use tool_choice: "any" reasoning - explain when you would use it here vs a forced tool.
explanation = "..."  # TODO
print(explanation)

# Expected output: A correct explanation of when 'any' vs. a forced tool choice applies to this extraction.

### Exercise 4: Retry loop for a semantic error

**Objective**: Handle a semantic validation error (not a syntax error) and decide if retry will help.

**Produce**: A decision on retry-worthiness plus the retry call if applicable.

In [ ]:
document = "Line items: $10 + $15 + $20. Stated total: $40."
extraction = {"line_items": [10, 15, 20], "stated_total": 40}

def calculated_total(extraction: dict) -> float:
    return sum(extraction["line_items"])

# TODO: check if calculated_total matches stated_total; if not, decide if retrying with feedback will help
# (hint: this is a semantic error the source data itself contains - will simply retrying fix it?)
mismatch = calculated_total(extraction) != extraction["stated_total"]
print(mismatch)
retry_will_help = None  # TODO
print(retry_will_help)

# Expected output: Correctly identifies the mismatch and reasons that retry will NOT help since the source document itself is inconsistent.

In [ ]:
# STRETCH GOAL: design the "conflict_detected" boolean field this schema should have included from the start.
improved_schema_snippet = {"conflict_detected": "boolean, true if calculated_total != stated_total"}
print(improved_schema_snippet)

# Expected output: A schema addition that would have flagged this conflict at extraction time instead of after the fact.

### Exercise 5: Batch vs. realtime decision

**Objective**: Decide batch vs. realtime for 4 workloads and calculate a submission schedule for one.

**Produce**: 4 decisions plus one scheduling calculation.

In [ ]:
workloads = [
    ("Pre-merge PR check", True, False),
    ("Weekly technical debt report", False, True),
    ("Live customer chat response", True, False),
    ("Nightly test generation for 500 files", False, True),
]
# TODO: call fits_batch_api(is_blocking, latency_tolerant) for each
for name, blocking, tolerant in workloads:
    pass

# TODO: if you need results within a 30-hour SLA and batches take up to 24h, how often must you submit?
sla_hours = 30
batch_window_hours = 24
submission_frequency_hours = None  # TODO
print(submission_frequency_hours)

# Expected output: Correct batch/realtime decisions, plus a submission frequency <= 6 hours to guarantee the 30h SLA.

In [ ]:
# STRETCH GOAL: simulate 3 failed documents by custom_id and write the resubmission batch.
failed_custom_ids = ["doc-1", "doc-7"]
resubmission_batch = [r for r in batch_requests if r["custom_id"] in failed_custom_ids]
print(resubmission_batch)

# Expected output: A resubmission batch containing only the failed documents, correlated by custom_id.

### Exercise 6: Multi-pass review design

**Objective**: Design a 2-pass review architecture for a 20-file PR and justify it over single-pass or self-review.

**Produce**: A described 2-pass architecture with justification.

In [ ]:
pr_file_count = 20

# TODO: describe pass 1 (per-file) and pass 2 (cross-file integration), and justify why this beats
# a single-pass review or a self-review by the same generating instance
pass_1 = "..."  # TODO
pass_2 = "..."  # TODO
justification = "..."  # TODO
print(pass_1, pass_2, justification, sep="\n")

# Expected output: A clear 2-pass design with justification referencing attention dilution and self-review limitations.

In [ ]:
# STRETCH GOAL: add a 3rd pass: an independent reviewer instance (no generation context) that re-checks
# the top 3 highest-severity findings from pass 2 before they're reported.
pass_3 = "..."  # TODO
print(pass_3)

# Expected output: A sensible third verification pass targeting only the highest-severity findings.

## Self-Assessment: Prompt Engineering & Structured Output

Answer each question, then check the Answers cell below.


**Q1.** Why do explicit, categorical criteria outperform vague instructions like 'only report high-confidence findings'?

**Q2.** What makes a few-shot example more effective at improving generalization: showing just the answer, or showing reasoning?

**Q3.** What does tool_use with a JSON schema guarantee, and what does it NOT guarantee?

**Q4.** When is a retry-with-feedback loop ineffective, no matter how well-designed the feedback is?

**Q5.** Why is a single-pass review of a 20-file PR prone to inconsistent, contradictory findings?

## Answers


**A1.** Because they give a specific, testable rule instead of relying on the model's self-reported confidence, which is not a reliable precision signal.

**A2.** Showing reasoning — it helps the model generalize judgment to novel patterns rather than only matching pre-specified cases.

**A3.** It guarantees schema-compliant syntax (no JSON errors); it does NOT guarantee semantic correctness (e.g., values that don't logically add up).

**A4.** When the needed information is simply absent from the source document — no amount of retrying can produce information that was never provided.

**A5.** Because processing many files at once causes attention dilution, leading to inconsistent depth and even contradictory findings across similar code.

---
**Next**: [Domain 5: Context Management & Reliability](./05-context-management-and-reliability.ipynb)
**Back**: [README](./README.md)